## Introduction

The goal of this notebook is to **compare different portfolio that gives exposition to 1 pricing parameter**.

## Import librairies

In [1]:
import numpy as np
import cvxpy as cp
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import itertools as it
import matplotlib.pyplot as plt
from sklearn import linear_model
from python_module.pricing_model import SABRModel

pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
pd.options.display.float_format = '{:,.2f}'.format

## Inputs

In [2]:
params = {
    
    'T0': {
        "F": 1000,
        "T": 5/250,
        "alpha": 0.1,
        "rho": -0.5,
        "nu": 0.5},

    'AlphaBump': {
        "F": 1000,
        "T": 5/250,
        "alpha": 0.11,
        "rho": -0.5,
        "nu": 0.5},
    
    'SkewBump': {
        "F": 1000,
        "T": 5/250,
        "alpha": 0.1,
        "rho": -0.0,
        "nu": 0.5},

    'VolOfVolBump': {
        "F": 1000,
        "T": 5/250,
        "alpha": 0.1,
        "rho": -0.5,
        "nu": 1}
    
}

slide_list = [0.01, -0.01, 0.05, -0.05]
slide_type = 'spot_only'

beta = 1
r = 0

## Generate Market data

In [3]:
# Generate options
results = dict()

for key in params.keys():
    globals().update(params[key])
    for K in range(940, 1060, 5):
        out = dict()
        option_type = 'call' if K > F else 'put'
        out = SABRModel.compute_option(F, K, T, alpha, beta, rho, nu, r, option_type=option_type, compute_bs_greeks=True, compute_model_greek=True,
                                        slide_list=slide_list, slide_type=slide_type)
        results[(key, K)] = out
        
results = pd.DataFrame(results).unstack().unstack()
results = results.reset_index(names=['index', 'strike'])
start_df = results[results['index']=='T0'].reset_index(drop=True)

In [4]:
start_df

,index,strike,IV,price,delta,gamma,vega,theta,vanna,volga,sabr_delta,sabr_gamma,sabr_vega,sabr_vanna,sabr_volga,sabr_theta,0.01,-0.01,0.05,-0.05
0,T0,940,0.11,0.00,-0.00,0.00,0.00,-0.00,-0.00,2.43,-0.00,0.00,0.02,-0.00,0.00,0.04,0.00,0.00,0.00,2.11
1,T0,945,0.11,0.00,-0.00,0.00,0.00,-0.00,-0.01,7.15,-0.00,0.00,0.06,-0.00,0.00,0.15,0.00,0.00,0.00,3.60
2,T0,950,0.11,0.00,-0.00,0.00,0.00,-0.00,-0.04,18.88,-0.00,0.00,0.18,-0.00,0.00,0.47,0.00,0.01,0.02,5.72
3,T0,955,0.11,0.00,-0.00,0.00,0.01,-0.01,-0.10,44.51,-0.00,0.00,0.50,-0.01,0.01,1.34,0.01,0.03,0.05,8.50
4,T0,960,0.11,0.01,-0.00,0.00,0.01,-0.01,-0.24,92.86,-0.00,0.00,1.30,-0.01,0.02,3.45,0.02,0.06,0.14,11.86
5,T0,965,0.10,0.04,-0.01,0.00,0.03,-0.03,-0.50,169.78,-0.01,0.00,3.07,-0.02,0.03,8.06,0.04,0.13,0.36,15.62
6,T0,970,0.10,0.10,-0.02,0.00,0.07,-0.07,-0.91,268.53,-0.02,0.00,6.51,-0.05,0.06,16.96,0.10,0.25,0.84,19.45
7,T0,975,0.10,0.24,-0.04,0.01,0.12,-0.13,-1.47,360.78,-0.04,0.01,12.40,-0.07,0.09,32.09,0.20,0.44,1.80,22.92
8,T0,980,0.10,0.54,-0.08,0.01,0.21,-0.22,-2.02,400.15,-0.08,0.01,21.18,-0.10,0.12,54.42,0.37,0.70,3.51,25.50
9,T0,985,0.10,1.09,-0.15,0.02,0.32,-0.33,-2.33,348.31,-0.14,0.02,32.31,-0.11,0.13,82.42,0.61,0.98,6.19,26.66


## 6. Compute variance P&L

In [100]:
pnl_results = dict()

for slide in slide_list:
    pnl_dict = dict()
    for col in ['weight', 'bucket', 'local_replication', 'brute_force', 'lasso_10', 'lasso_4', 'miqp', 'miqp_v2']:
        pnl_dict[col] = start_df[slide].dot(start_df[col].fillna(0))
    pnl_results[f'slide {slide*100:.0F}%'] = pnl_dict

for key in params.keys():
    temp_df = results[results['index']==key].reset_index(drop=True)
    pnl = temp_df.set_index('strike')['price'] - start_df.set_index('strike')['price']
    start_df[f'{key}_pnl'] = start_df['strike'].map(pnl)
    pnl_dict = dict()
    for col in ['weight', 'bucket', 'local_replication', 'brute_force', 'lasso_10', 'lasso_4', 'miqp', 'miqp_v2']:
        pnl_dict[col] = start_df[f'{key}_pnl'].dot(start_df[col].fillna(0))
    pnl_results[key] = pnl_dict

pnl_results = pd.DataFrame(pnl_results)

In [101]:
pnl_results * 1_000_000

,slide 1%,slide -1%,slide 5%,slide -5%,T0,T1,AlphaUp,AlphaDown,SkewFlat,SkewSteep,VolOfVolDown,VolOfVolUp
weight,"198,989.39","201,652.50","4,781,305.17","5,130,621.83",0.00,"-80,360.48","500,527.86","-300,751.21",-1.27,-1.52,-962.99,"3,036.58"
bucket,"195,239.26","192,829.93","3,122,072.50","3,019,990.26",0.00,"-82,717.77","420,894.56","-347,023.19",778.09,-887.32,132.17,477.94
local_replication,"201,490.02","200,019.60","3,761,530.09","4,566,418.46",0.00,"-79,343.90","486,028.64","-254,477.14",645.38,-835.39,-481.32,"2,156.53"
brute_force,"197,767.38","203,057.68","4,783,112.36","5,181,856.04",0.00,"-80,610.50","499,735.68","-286,230.24",-496.47,344.42,"-1,336.15","3,447.19"
lasso_10,"200,309.43","200,491.63","4,804,487.12","5,119,923.40",0.00,"-79,805.49","501,086.97","-277,922.94",395.86,-411.07,-675.41,"2,604.73"
lasso_4,"208,639.85","198,069.15","5,372,431.89","4,865,740.78",0.00,"-77,499.71","529,100.77","-256,802.64","1,868.56","-1,540.98",232.39,"2,061.80"
miqp,"197,894.95","198,277.98","4,786,129.76","5,117,750.30",0.00,"-77,627.53","500,277.53","-244,218.18",193.32,-353.79,-904.18,"2,961.56"
miqp_v2,"200,575.56","201,652.50","4,825,802.51","5,192,429.07",0.00,"-78,849.41","507,499.50","-248,089.87",85.93,-274.19,"-1,004.84","3,107.72"


In [102]:
(pnl_results - pnl_results.loc['weight']) * 1_000_000

,slide 1%,slide -1%,slide 5%,slide -5%,T0,T1,AlphaUp,AlphaDown,SkewFlat,SkewSteep,VolOfVolDown,VolOfVolUp
weight,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
bucket,"-3,750.12","-8,822.57","-1,659,232.68","-2,110,631.56",0.00,"-2,357.28","-79,633.30","-46,271.98",779.36,-885.79,"1,095.16","-2,558.64"
local_replication,"2,500.63","-1,632.89","-1,019,775.09","-564,203.37",0.00,"1,016.58","-14,499.22","46,274.07",646.66,-833.86,481.67,-880.05
brute_force,"-1,222.01","1,405.19","1,807.18","51,234.22",0.00,-250.01,-792.18,"14,520.97",-495.20,345.94,-373.16,410.61
lasso_10,"1,320.05","-1,160.87","23,181.95","-10,698.42",0.00,554.99,559.11,"22,828.27",397.13,-409.55,287.58,-431.85
lasso_4,"9,650.46","-3,583.34","591,126.72","-264,881.05",0.00,"2,860.78","28,572.90","43,948.57","1,869.83","-1,539.46","1,195.38",-974.78
miqp,"-1,094.43","-3,374.51","4,824.59","-12,871.53",0.00,"2,732.95",-250.33,"56,533.03",194.59,-352.26,58.81,-75.02
miqp_v2,"1,586.18",0.00,"44,497.34","61,807.25",0.00,"1,511.08","6,971.64","52,661.35",87.20,-272.67,-41.85,71.14


In [106]:
(pnl_results - pnl_results.loc['weight']).min(axis=1).sort_values(ascending=False) * 1_000_000

weight                       0.00
miqp_v2                   -272.67
brute_force             -1,222.01
lasso_10               -10,698.42
miqp                   -12,871.53
lasso_4               -264,881.05
local_replication   -1,019,775.09
bucket              -2,110,631.56
dtype: float64

In [ ]:
(pnl_results - pnl_results.loc['weight']).mean(axis=1).sort_values(ascending=False)  * 1_000_000

lasso_4               34,020.50
miqp_v2               14,073.22
brute_force            5,549.30
miqp                   3,860.49
lasso_10               3,035.70
weight                     0.00
local_replication   -129,242.07
bucket              -326,022.45
dtype: float64

In [113]:
sharpe = (pnl_results - pnl_results.loc['weight']).mean(axis=1).sort_values(ascending=False) / (pnl_results - pnl_results.loc['weight']).std(axis=1).sort_values(ascending=False) 
sharpe.sort_values(ascending=False).dropna()

miqp_v2              0.59
brute_force          0.37
lasso_10             0.31
miqp                 0.23
lasso_4              0.18
local_replication   -0.40
bucket              -0.44
dtype: float64